# IRIS — Neural IDS for AI Agent Pipelines

**One-click launcher** for the IRIS interactive dashboard.

This notebook clones the project from GitHub, installs dependencies, downloads pre-trained checkpoints, and launches the Gradio web app with a **public URL** you can open in any browser.

### What you need
- A Google Colab runtime with a **T4 GPU** (free tier works)
- ~60 seconds for model loading after setup

### Instructions
Select `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

*Nathan Cheung (ncheung3@my.yorku.ca) | York University | CSSD 2221 | Winter 2026*

In [ ]:
# === Mount Google Drive (for caching) and clone repository ===
from google.colab import drive
drive.mount('/content/drive')

# Always fetch latest code from GitHub
import shutil, os
os.chdir('/content')  # ensure valid working directory before cleanup
if os.path.exists('/content/iris'):
    shutil.rmtree('/content/iris')
!git clone https://github.com/ncheung13579/iris.git /content/iris
%cd /content/iris

# transformer-lens pins numpy<2 but works fine with numpy 2.x (pure Python).
# Install it without deps to avoid downgrading Colab's numpy, which would
# require a runtime restart. Its real deps are in requirements.txt.
!pip install transformer-lens --no-deps -q
!grep -v 'transformer-lens' requirements.txt | pip install -r /dev/stdin -q
print('Setup complete.')

In [ ]:
# === Download pre-trained checkpoints (with Drive cache) ===
# First run: downloads from GitHub Release → caches to Drive.
# Subsequent runs: restores from Drive instantly, even after runtime reset.
# If Drive is full, caching is skipped — the download still works.
import os, shutil, urllib.request

RELEASE_URL = "https://github.com/ncheung13579/iris/releases/download/v1.0.0"
DRIVE_CACHE = "/content/drive/MyDrive/iris_checkpoints"

files = {
    "checkpoints/sae_d10240_lambda1e-04.pt": "sae_d10240_lambda1e-04.pt",
    "checkpoints/feature_matrix.npy": "feature_matrix.npy",
    "checkpoints/sensitivity_scores.npy": "sensitivity_scores.npy",
    "data/processed/iris_dataset_balanced.json": "iris_dataset_balanced.json",
}

os.makedirs(DRIVE_CACHE, exist_ok=True)

for local_path, remote_name in files.items():
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    drive_path = os.path.join(DRIVE_CACHE, remote_name)

    if os.path.exists(local_path):
        # Already in the working directory (re-run within same runtime)
        print(f"  [OK] {local_path} (already exists)")
    elif os.path.exists(drive_path):
        # Restore from Drive cache (runtime recycled but Drive persists)
        shutil.copy2(drive_path, local_path)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"  [OK] {local_path} (restored from Drive, {size_mb:.1f} MB)")
    else:
        # First time: download from GitHub Release
        url = f"{RELEASE_URL}/{remote_name}"
        print(f"  Downloading {remote_name}...", end=" ", flush=True)
        urllib.request.urlretrieve(url, local_path)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        # Cache to Drive (best-effort — skip if Drive is full)
        try:
            shutil.copy2(local_path, drive_path)
            print(f"OK ({size_mb:.1f} MB, cached to Drive)")
        except OSError:
            print(f"OK ({size_mb:.1f} MB, Drive cache skipped — low storage)")

print("\nAll checkpoints ready.")

In [ ]:
# === Launch the IRIS Detection Dashboard ===
# This loads GPT-2, the SAE, and all detectors (~60 seconds),
# then auto-opens the dashboard in a new browser tab.
from src.app import IRISPipeline, build_app

pipeline = IRISPipeline('.')
pipeline.load()
app = build_app(pipeline)

# Suppress Colab's automatic inline rendering by temporarily
# blocking IPython display calls during launch
import IPython.display as _ipd
_original_display = _ipd.display
_ipd.display = lambda *a, **kw: None

try:
    _, local_url, share_url = app.launch(
        share=True,
        prevent_thread_lock=True,
    )
finally:
    _ipd.display = _original_display

url = share_url or local_url

# Show a clickable link + auto-open via JavaScript
_ipd.display(_ipd.HTML(
    f'<script>window.open("{url}", "_blank");</script>'
    f'<div style="padding:16px; background:#f0f9ff; border:2px solid #2563EB; '
    f'border-radius:10px; margin:10px 0; font-size:16px; text-align:center;">'
    f'<b>IRIS Dashboard is live!</b><br><br>'
    f'If it didn\'t open automatically, '
    f'<a href="{url}" target="_blank" style="font-size:18px; color:#2563EB;">'
    f'click here to open IRIS</a><br><br>'
    f'<code style="font-size:13px; opacity:0.7;">{url}</code></div>'
))